# RUN FULL PIPELINE

News in, a portfolio and its track record out. One cell per stage, each returning the frame
so it can be looked at before the next one runs.

```
  corpus ──▶ detect ──▶ themes ──▶ monitor ──▶ signal ──▶ basket ──▶ weights ──▶ backtest
```

`ASOF` is the day the pipeline pretends it is: detection and the basket never read past it.
`UNTIL` is how far the monitor is then run forward, which is the only place the future enters —
and it may, because the score of each week is computed from the weeks before it.

The same run from a terminal: `python RUN_FULL_PIPELINE.py --asof 2023-01-02 --until 2025-12-31`


In [ ]:
import importlib, sys
sys.path.insert(0, '.')
import polars as pl
import config, pipeline, artifacts

ASOF, UNTIL = '2023-01-16', '2025-12-31'   # 01-16, not 01-02: gate 3 needs two weeks of persistence

config.load_env()
pipeline.check_environment()


## 1 · Detect — which words became a theme

Four gates over the five months ending at `ASOF`, against twenty-four months of baseline.
Slow: it reads the whole corpus and calls the model once per surviving cluster.


In [ ]:
detect = pipeline.stage('detect')
corpus = pl.scan_parquet(config.CORPUS)

themes, theme_diag, rejects = detect.detect(corpus, ASOF, **pipeline.params('detect'))
themes


## 2 · Monitor — when to be in, and when to be out

Two halves that do not share a measure: `entry/` opens on ACCELERATION, `exit/` closes on LEVEL.
Run forward to `UNTIL`, with the window widened so it still contains each theme's birth.


In [ ]:
monitor = pipeline.stage('monitor')

signal, signal_diag = monitor.monitor(
    corpus, themes, UNTIL, **pipeline.monitor_params(themes, ASOF, UNTIL))
monitor.entry_dates(signal)


In [ ]:
import plotly.graph_objects as go

d = signal_diag.join(signal.select('theme', 'week', 'position'), on=['theme', 'week']).sort('week')
fig = go.Figure()
for theme in d['theme'].unique().sort():
    t = d.filter(pl.col('theme') == theme)
    fig.add_scatter(x=t['week'], y=t['hits'], name=f'{theme} · headlines/week')
    held = t.filter('position')
    if len(held):
        fig.add_vrect(x0=held['week'].min(), x1=held['week'].max(),
                      fillcolor='LightSalmon', opacity=.25, line_width=0)
fig.update_layout(height=340, margin=dict(t=30, b=20),
                  title='weekly volume; shaded = holding the theme')
fig.show()


## 3 · Basket — which companies those words buy

A model turns each of `vocab_wide`'s brands into the company that owns it, and that company
is looked up in the SEC register: one holding per cik, equal weight.


In [ ]:
basket = pipeline.stage('basket')

weights, weight_diag = basket.weights(themes, **pipeline.params('basket'))
weight_diag


## 4 · Backtest — what that was worth

Hold the basket while `position` is true, flat otherwise, trading the first price date after
the week boundary. The window stops where the signal stops, never where the prices do.


In [ ]:
prices = importlib.import_module('4_backtesting.prices')
rebalance = pipeline.stage('backtest')
params = pipeline.params('backtest')

quotes = prices.load(sorted({t.upper() for t in weights['ticker']} | {params['benchmark']}),
                     max_age_days=params.pop('price_max_age_days'))
performance, curves = rebalance.backtest(weights, signal, quotes, **params)
performance


In [ ]:
fig = go.Figure()
for theme in curves['theme'].unique().sort():
    c = curves.filter(pl.col('theme') == theme).sort('date')
    fig.add_scatter(x=c['date'], y=c['level'], name=theme)
    fig.add_scatter(x=c['date'], y=c['bench'], name=f'{params["benchmark"]} · same window',
                    line=dict(dash='dot'))
fig.update_layout(height=380, margin=dict(t=30, b=20), yaxis_title='level, 1.0 at entry',
                  title='the theme against its benchmark')
fig.show()


## 5 · Save the run

Four stages, four directories, all named after the run id — so two runs can be compared.
`pipeline.run(ASOF, until=UNTIL)` does every cell above and saves in one call.


In [ ]:
for stage, frame, diag in [('themes', themes, theme_diag), ('signal', signal, signal_diag),
                          ('weights', weights, weight_diag), ('backtest', performance, curves)]:
    artifacts.save(stage, frame, diag, asof=ASOF, until=UNTIL)

{s: artifacts.runs(s) for s in ('themes', 'signal', 'weights', 'backtest')}
